# Comparação de Modelos: MLP vs Baselines

Este notebook compara o desempenho de:
1. **DummyClassifier**: Baseline trivial
2. **LogisticRegression**: Baseline linear
3. **MLPClassifier**: Rede neural em PyTorch

## Métricas Utilizadas
- **Accuracy**: Acurácia geral
- **Precision**: Taxa de acertos entre os positivos previstos
- **Recall**: Taxa de acertos entre os positivos reais
- **F1-score**: Média harmônica de precision e recall
- **ROC-AUC**: Área sob a curva ROC
- **PR-AUC**: Área sob a curva Precision-Recall
- **Cost Saved**: Economia em reais ($) evitando churns
- **ROI**: Retorno sobre investimento

In [ ]:
import logging
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from projeto_ml import train, data as data_mod, features as features_mod, eval as eval_mod
from sklearn.model_selection import StratifiedKFold

logging.basicConfig(level=logging.INFO)
LOGGER = logging.getLogger(__name__)

# Configurar estilo de plots
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

## 1. Carregar e Preparar Dados

In [ ]:
# Carregar dados
dataset_path = "data/raw/Telco_customer_churn(Telco_Churn).csv"
df = data_mod.load_telco_churn(dataset_path, drop_leakage=True)

print(f"Dataset shape: {df.shape}")
print(f"\nClass distribution:")
print(df[data_mod.TARGET_COLUMN].value_counts())
print(f"\nClass balance: {df[data_mod.TARGET_COLUMN].value_counts(normalize=True).to_dict()}")

## 2. Executar Treinamento com Todos os Modelos

In [ ]:
# Treinar todos os modelos
LOGGER.info("Iniciando treinamento com todos os modelos...")

result = train.train(
    dataset_path=dataset_path,
    model_out=Path("models/all_models_final.joblib"),
    n_splits=5,
    metric="roc_auc",
    cost_per_churn=100.0,
    intervention_success=0.3,
    models=["dummy", "logistic", "mlp"],
)

LOGGER.info("Treinamento concluído!")
print("\nResultados:")
for key, value in sorted(result['metrics'].items()):
    print(f"  {key}: {value:.6f}")

## 3. Agregar Resultados por Modelo

In [ ]:
# Organizar resultados por modelo
metrics_by_model = {}

for key, values in result['metrics'].items():
    model_name, metric_name = key.rsplit('_', 1)
    
    if model_name not in metrics_by_model:
        metrics_by_model[model_name] = {}
    
    metrics_by_model[model_name][metric_name] = values

# Criar tabela resumida
summary_rows = []

for model_name, metrics in metrics_by_model.items():
    row = {"Model": model_name}
    for metric_name, value in metrics.items():
        row[f"{metric_name}_mean"] = np.nanmean(value)
        row[f"{metric_name}_std"] = np.nanstd(value)
    summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows)
print("\nResumo de Resultados por Modelo:")
print(df_summary.to_string(index=False))

## 4. Tabela Comparativa Final

In [ ]:
# Criar tabela com principais métricas
comparison_data = []

for model_name, metrics in metrics_by_model.items():
    comparison_data.append({
        "Model": model_name.upper(),
        "Accuracy": f"{np.nanmean(metrics.get('accuracy', [0])):.4f}",
        "Precision": f"{np.nanmean(metrics.get('precision', [0])):.4f}",
        "Recall": f"{np.nanmean(metrics.get('recall', [0])):.4f}",
        "F1-Score": f"{np.nanmean(metrics.get('f1', [0])):.4f}",
        "ROC-AUC": f"{np.nanmean(metrics.get('roc_auc', [0])):.4f}",
        "PR-AUC": f"{np.nanmean(metrics.get('pr_auc', [0])):.4f}",
        "Cost Saved ($)": f"{np.nanmean(metrics.get('cost_saved', [0])):.2f}",
    })

df_comparison = pd.DataFrame(comparison_data)
print("\n" + "="*100)
print("COMPARAÇÃO FINAL DE MODELOS")
print("="*100)
print(df_comparison.to_string(index=False))
print("="*100)

## 5. Visualização de Métricas

In [ ]:
# Preparar dados para visualização
viz_data = []

for model_name, metrics in metrics_by_model.items():
    for metric_name in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'pr_auc']:
        if metric_name in metrics:
            for fold_idx, value in enumerate(metrics[metric_name], 1):
                viz_data.append({
                    'Model': model_name.upper(),
                    'Métrica': metric_name.upper().replace('_', '-'),
                    'Valor': value,
                    'Fold': fold_idx
                })

df_viz = pd.DataFrame(viz_data)

# Plot: Distribuição de métricas por modelo
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

metrics_to_plot = ['ACCURACY', 'PRECISION', 'RECALL', 'F1-SCORE', 'ROC-AUC', 'PR-AUC']

for idx, metric in enumerate(metrics_to_plot):
    data_subset = df_viz[df_viz['Métrica'] == metric]
    sns.boxplot(data=data_subset, x='Model', y='Valor', ax=axes[idx], palette='Set2')
    axes[idx].set_title(f'{metric} por Modelo', fontsize=12, fontweight='bold')
    axes[idx].set_ylabel('Valor')
    axes[idx].set_ylim([0, 1])
    axes[idx].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('models/comparison_metrics.png', dpi=300, bbox_inches='tight')
print("Plot salvo em: models/comparison_metrics.png")
plt.show()

## 6. Análise de Custo-Benefício

In [ ]:
# Análise de custo salvo por modelo
cost_data = []

for model_name, metrics in metrics_by_model.items():
    if 'cost_saved' in metrics:
        cost_data.append({
            'Model': model_name.upper(),
            'Cost Saved (Mean)': np.nanmean(metrics['cost_saved']),
            'Cost Saved (Std)': np.nanstd(metrics['cost_saved']),
            'Cost Saved (Min)': np.nanmin(metrics['cost_saved']),
            'Cost Saved (Max)': np.nanmax(metrics['cost_saved']),
        })

df_cost = pd.DataFrame(cost_data)
print("\nAnálise de Custo Salvo ($):")
print(df_cost.to_string(index=False))

# Plot: Comparação de economia
fig, ax = plt.subplots(figsize=(10, 6))
models = df_cost['Model'].values
means = df_cost['Cost Saved (Mean)'].values
stds = df_cost['Cost Saved (Std)'].values

bars = ax.bar(models, means, yerr=stds, capsize=10, alpha=0.7, color=['#FF6B6B', '#4ECDC4', '#45B7D1'])
ax.set_ylabel('Economia ($)', fontsize=12, fontweight='bold')
ax.set_xlabel('Modelo', fontsize=12, fontweight='bold')
ax.set_title('Custo Salvo com Prevenção de Churn por Modelo', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# Adicionar valores nas barras
for bar, mean in zip(bars, means):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'${mean:.2f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('models/cost_analysis.png', dpi=300, bbox_inches='tight')
print("\nPlot salvo em: models/cost_analysis.png")
plt.show()

## 7. Conclusões e Recomendações

In [ ]:
# Identificar melhor modelo por métrica
print("\n" + "="*100)
print("MELHOR MODELO POR MÉTRICA")
print("="*100)

metrics_comparison = {}

for metric_name in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'pr_auc', 'cost_saved']:
    best_model = None
    best_value = -np.inf
    
    for model_name, metrics in metrics_by_model.items():
        if metric_name in metrics:
            mean_value = np.nanmean(metrics[metric_name])
            if mean_value > best_value:
                best_value = mean_value
                best_model = model_name
    
    print(f"\n{metric_name.upper().replace('_', ' ')}:")
    print(f"  Melhor modelo: {best_model.upper()}")
    print(f"  Valor: {best_value:.4f}")

print("\n" + "="*100)

## 8. Salvar Tabela Comparativa

In [ ]:
# Salvar tabela comparativa em CSV
output_path = Path("models/model_comparison_results.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

# Criar DataFrame detalhado
detailed_results = []

for model_name, metrics in metrics_by_model.items():
    for metric_name in metrics.keys():
        values = metrics[metric_name]
        detailed_results.append({
            'Model': model_name.upper(),
            'Metric': metric_name,
            'Mean': np.nanmean(values),
            'Std': np.nanstd(values),
            'Min': np.nanmin(values),
            'Max': np.nanmax(values),
        })

df_detailed = pd.DataFrame(detailed_results)
df_detailed.to_csv(output_path, index=False)

print(f"Tabela comparativa salva em: {output_path}")
print(f"\nResultados salvos:")
print(df_detailed.to_string(index=False))